# 🇹🇭 Fine-tune FunctionGemma-270m-it for Thai Tool Calling (v9)

**Root cause fix**: v7/v8 trained the model to output FunctionGemma's
`<start_function_call>` format, but the model's built-in Jinja template
expects **JSON output**:
- Tool call: `{"tool_call": {"name": "func", "arguments": {...}}}`
- Response: `{"response": "Thai text"}`

v9 trains with the **exact format** the server sends + expects.


## 1. Install Dependencies

In [ ]:
# Note: bitsandbytes is NOT included — it doesn't support Apple Silicon (MPS).
# This notebook auto-detects CUDA / MPS (Apple Silicon) / CPU.
!pip install -q -U \
    torch \
    transformers \
    datasets \
    peft \
    trl \
    accelerate \
    openai \
    tqdm \
    matplotlib

In [ ]:
import torch
import json
import os
import time
import re
from pathlib import Path
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoProcessor,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer
from openai import OpenAI
from tqdm.auto import tqdm

print(f"PyTorch version: {torch.__version__}")

# ═══════════════════════════════════════════════════════════════
# Auto-detect device: CUDA > MPS (Apple Silicon) > CPU
# ═══════════════════════════════════════════════════════════════
if torch.cuda.is_available():
    DEVICE = 'cuda'
    print(f"✅ Using CUDA GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = 'mps'
    print(f"✅ Using Apple Silicon MPS (Metal Performance Shaders)")
else:
    DEVICE = 'cpu'
    print(f"⚠️ Using CPU (training will be slower)")

print(f"Device: {DEVICE}")

In [ ]:
HF_TOKEN="<YOUR_HF_TOKEN>"

## 2. Verify Translation Endpoint

We connect to `gemma-3-27b-it` running on LM Studio to translate the dataset to Thai.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Configuration
# ═══════════════════════════════════════════════════════════════
TRANSLATION_ENDPOINT = "http://<LLM_ENDPOINT_HOST>:1234/v1"
API_KEY = "lm-studio"

# Connect to LM Studio
client = OpenAI(base_url=TRANSLATION_ENDPOINT, api_key=API_KEY)

# Discover available models
try:
    models = client.models.list()
    available_models = [m.id for m in models.data]
    print(f"✅ Connected! Available models: {available_models}")
    
    # Use the first available model (or specify explicitly)
    TRANSLATION_MODEL = available_models[0] if available_models else "gemma-3-27b-it"
    print(f"Using model: {TRANSLATION_MODEL}")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("Please ensure LM Studio is running at the specified endpoint.")
    TRANSLATION_MODEL = "gemma-3-27b-it"

In [ ]:
# Quick translation test
test_response = client.chat.completions.create(
    model=TRANSLATION_MODEL,
    messages=[
        {
            "role": "user",
            "content": (
                "Translate the following English text to Thai. "
                "Output ONLY the Thai translation, nothing else.\n\n"
                "Text: Rename the file /mnt/storage to /backup/docs."
            ),
        }
    ],
    temperature=0.3,
    max_tokens=256,
)

print(f"🇹🇭 Test translation: {test_response.choices[0].message.content.strip()}")

## 3. Load & Explore Datasets

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Load Dataset 1: Canfield/tool-calls-dataset
# ═══════════════════════════════════════════════════════════════
raw_canfield = load_dataset("Canfield/tool-calls-dataset", split="train")
print(f"Canfield dataset: {len(raw_canfield)} examples")
print(f"Columns: {raw_canfield.column_names}")
print()

# ═══════════════════════════════════════════════════════════════
# Load Dataset 2: google/mobile-actions
# ═══════════════════════════════════════════════════════════════
raw_mobile = load_dataset("google/mobile-actions", split="train")
print(f"Mobile-actions dataset: {len(raw_mobile)} examples")
print(f"Columns: {raw_mobile.column_names}")
print()
print(f"Total combined: {len(raw_canfield) + len(raw_mobile)} examples")


### 3.3 Load Home Assistant Dataset

Load `acon96/Home-Assistant-Requests-V2` — a smart home tool-calling dataset.

Contains ~209K examples of controlling Home Assistant devices (lights, covers,
climate, media, vacuums, timers, etc.) with 19 HA service tools.
We sample 5,000 to keep training balanced.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Load Dataset 3: acon96/Home-Assistant-Requests-V2
# ═══════════════════════════════════════════════════════════════
raw_homeassistant_full = load_dataset(
    "acon96/Home-Assistant-Requests-V2", split="train"
)
print(f"Home Assistant full dataset: {len(raw_homeassistant_full)} examples")
print(f"Columns: {raw_homeassistant_full.column_names}")

# Sample down to keep training balanced
HA_SAMPLE_SIZE = 5000
raw_homeassistant = raw_homeassistant_full.shuffle(seed=42).select(
    range(min(HA_SAMPLE_SIZE, len(raw_homeassistant_full)))
)
print(f"\nSampled: {len(raw_homeassistant)} examples")
print(f"\nTotal combined: {len(raw_canfield) + len(raw_mobile) + len(raw_homeassistant)} examples")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Inspect Home Assistant examples
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print("HOME ASSISTANT SAMPLES")
print("=" * 60)
for i in range(min(2, len(raw_homeassistant))):
    example = raw_homeassistant[i]
    print(f"\n--- Home Assistant Example {i} ---")

    # Show tools
    tools = example.get("tools", [])
    print(f"  Tools ({len(tools)}):")
    for t in tools[:5]:
        func = t.get("function", t)
        print(f"    - {func.get('name', '?')}: {str(func.get('description', ''))[:80]}")
    if len(tools) > 5:
        print(f"    ... and {len(tools) - 5} more")

    # Show messages
    msgs = example.get("messages", [])
    for msg in msgs:
        role = msg.get("role", "?")
        content_list = msg.get("content", [])
        # Content is a list of {type, text} dicts
        if isinstance(content_list, list):
            text = " ".join(c.get("text", "") for c in content_list if c.get("text"))[:120]
        else:
            text = str(content_list)[:120]
        tc = msg.get("tool_calls", [])
        if text:
            print(f"  [{role}]: {text}")
        if tc:
            print(f"  [{role} tool_calls]: {str(tc)[:150]}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Inspect Canfield examples
# ═══════════════════════════════════════════════════════════════
print("=" * 60)
print("CANFIELD SAMPLES")
print("=" * 60)
for i in range(min(2, len(raw_canfield))):
    msgs = raw_canfield[i]["message"]
    if isinstance(msgs, str):
        msgs = json.loads(msgs)
    print(f"\n--- Canfield Example {i} ---")
    for turn in msgs:
        role = turn.get("role", "?")
        content = str(turn.get("content", ""))[:120]
        print(f"  [{role}]: {content}")

# ═══════════════════════════════════════════════════════════════
# Inspect mobile-actions examples
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("MOBILE-ACTIONS SAMPLES")
print("=" * 60)
for i in range(min(2, len(raw_mobile))):
    example = raw_mobile[i]
    print(f"\n--- Mobile-Actions Example {i} ---")

    # Show tools
    tools = example.get("tools", [])
    if isinstance(tools, str):
        tools = json.loads(tools)
    print(f"  Tools ({len(tools)}):")
    for t in tools[:3]:
        func = t.get("function", t)
        print(f"    - {func.get('name', '?')}: {str(func.get('description', ''))[:80]}")

    # Show messages
    msgs = example.get("messages", [])
    if isinstance(msgs, str):
        msgs = json.loads(msgs)
    for turn in msgs:
        role = turn.get("role", "?")
        content = str(turn.get("content", ""))[:120]
        tc = turn.get("tool_calls", [])
        if content:
            print(f"  [{role}]: {content}")
        if tc:
            print(f"  [{role} tool_calls]: {str(tc)[:150]}")


## 3.5 Normalize mobile-actions to Common Format

Convert `google/mobile-actions` to the same `[{role, content}]` format as Canfield.

- **User message**: `messages[0]["content"]`
- **Tool call**: `messages[1]["tool_calls"][0]["function"]` → `{"name": ..., "arguments": ...}`
- **Tool list**: Extracted from the `tools` field (names, descriptions, param names)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Normalize mobile-actions into Canfield-like format
# ═══════════════════════════════════════════════════════════════

def extract_mobile_action_tools(tools_field):
    """
    Extract tool info from mobile-actions 'tools' field.
    Returns list of dicts: [{name, args, description}, ...]
    """
    if isinstance(tools_field, str):
        tools_field = json.loads(tools_field)
    result = []
    for t in tools_field:
        func = t.get("function", t)
        name = func.get("name", "")
        desc = func.get("description", "")
        params = func.get("parameters", {})
        # Extract parameter names from the JSON schema
        props = params.get("properties", {})
        arg_names = sorted(props.keys())
        result.append({"name": name, "args": arg_names, "description": desc})
    return result


def normalize_mobile_action(example):
    """
    Convert a single mobile-actions example to our common format:
    Returns: (conversation_turns, tool_info_list) or (None, None)
    
    conversation_turns = [{"role": "user", "content": ...}, {"role": "assistant", "content": json_str}]
    tool_info_list = [{"name": ..., "args": [...], "description": ...}, ...]
    """
    messages = example.get("messages", [])
    if isinstance(messages, str):
        messages = json.loads(messages)

    tools = extract_mobile_action_tools(example.get("tools", []))

    user_content = None
    tool_call_json = None

    for msg in messages:
        role = msg.get("role", "")
        if role == "user":
            user_content = msg.get("content", "")
        elif role == "assistant":
            tc_list = msg.get("tool_calls", [])
            if isinstance(tc_list, str):
                tc_list = json.loads(tc_list)
            if tc_list:
                tc = tc_list[0]  # Take first tool call
                func = tc.get("function", {})
                fn_name = func.get("name", "")
                fn_args = func.get("arguments", "{}")
                if isinstance(fn_args, str):
                    try:
                        fn_args = json.loads(fn_args)
                    except json.JSONDecodeError:
                        fn_args = {}
                tool_call_json = json.dumps({"name": fn_name, "arguments": fn_args}, default=str)

    if user_content and tool_call_json:
        conversation = [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": tool_call_json},
        ]
        return conversation, tools
    return None, None


# Normalize all mobile-actions examples
mobile_conversations = []  # list of (conversation, tool_info_list)
mobile_skipped = 0

for i in range(len(raw_mobile)):
    conv, tools = normalize_mobile_action(raw_mobile[i])
    if conv is not None:
        mobile_conversations.append((conv, tools))
    else:
        mobile_skipped += 1

print(f"✅ Normalized {len(mobile_conversations)} mobile-actions examples ({mobile_skipped} skipped)")
print()

# Show a sample
if mobile_conversations:
    sample_conv, sample_tools = mobile_conversations[0]
    print("Sample normalized mobile-action:")
    print(f"  User: {sample_conv[0]['content']}")
    print(f"  Tool call: {sample_conv[1]['content'][:120]}")
    print(f"  Available tools: {[t['name'] for t in sample_tools]}")


## 3.6 Normalize Home Assistant Data to Common Format

Convert `acon96/Home-Assistant-Requests-V2` to the same `[{role, content}]` format.

The HA dataset has a richer message structure:
- **System**: Long context with device states (we skip this for the user prompt)
- **User**: `messages[1]["content"][0]["text"]` — the actual request
- **Assistant**: text response + `tool_calls[0]["function"]` → `{name, arguments}`
- **Tool list**: Extracted from the `tools` field (same approach as mobile-actions)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Normalize Home Assistant data into common format
# ═══════════════════════════════════════════════════════════════

def extract_ha_tools(tools_field):
    """
    Extract tool info from HA 'tools' field.
    Returns list of dicts: [{name, args, description}, ...]
    """
    result = []
    for t in tools_field:
        func = t.get("function", t)
        name = func.get("name", "")
        desc = func.get("description", "")
        params = func.get("parameters", {})
        # Extract parameter names — filter out nulls
        props = params.get("properties", {})
        arg_names = sorted(k for k, v in props.items() if v is not None)
        result.append({"name": name, "args": arg_names, "description": desc})
    return result


def normalize_homeassistant(example):
    """
    Convert a single HA example to our common format.
    Returns: (conversation_turns, tool_info_list) or (None, None)

    Conversation format:
      [{"role": "user", "content": "..."}, {"role": "assistant", "content": json_str}]
    """
    messages = example.get("messages", [])
    tools = extract_ha_tools(example.get("tools", []))

    user_content = None
    tool_call_json = None

    for msg in messages:
        role = msg.get("role", "")

        if role == "user":
            # Content is a list of {type, text}
            content_list = msg.get("content", [])
            if isinstance(content_list, list):
                texts = [c.get("text", "") for c in content_list if c.get("text")]
                user_content = " ".join(texts).strip()
            else:
                user_content = str(content_list).strip()

        elif role == "assistant":
            # Get the first tool call if present
            tc_list = msg.get("tool_calls", []) or []
            if tc_list and not tool_call_json:
                tc = tc_list[0]
                func = tc.get("function", {})
                fn_name = func.get("name", "")
                fn_args_str = func.get("arguments", "{}")
                if isinstance(fn_args_str, str):
                    try:
                        fn_args = json.loads(fn_args_str)
                    except json.JSONDecodeError:
                        fn_args = {}
                elif isinstance(fn_args_str, dict):
                    fn_args = fn_args_str
                else:
                    fn_args = {}
                tool_call_json = json.dumps({"name": fn_name, "arguments": fn_args})

    if user_content and tool_call_json:
        conversation = [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": tool_call_json},
        ]
        return conversation, tools
    return None, None


# Normalize all HA examples
ha_conversations = []  # list of (conversation, tool_info_list)
ha_skipped = 0

for i in range(len(raw_homeassistant)):
    conv, tools = normalize_homeassistant(raw_homeassistant[i])
    if conv is not None:
        ha_conversations.append((conv, tools))
    else:
        ha_skipped += 1

print(f"✅ Normalized {len(ha_conversations)} Home Assistant examples ({ha_skipped} skipped)")
print()

# Show a sample
if ha_conversations:
    sample_conv, sample_tools = ha_conversations[0]
    print("Sample normalized Home Assistant example:")
    print(f"  User: {sample_conv[0]['content']}")
    print(f"  Tool call: {sample_conv[1]['content'][:120]}")
    print(f"  Available tools ({len(sample_tools)}): {[t['name'] for t in sample_tools[:5]]}")


## 3.7 Generate Thai General Conversation Data

We generate synthetic Thai conversations using Gemma 3 so the model learns
**when NOT to call a tool** — i.e., when the user is just chatting.

**Rules for generated conversations:**
- Self-contained (no external data needed — no weather, no location lookups)
- Cover: greetings, opinions, explanations, math, language, culture, advice, humor
- Model responds with **normal text** (NOT a tool call)
- ~1,000 examples to balance against ~10,654 tool-calling examples (~10%)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Generate Thai general conversation data via Gemma 3
# ═══════════════════════════════════════════════════════════════
GENERAL_CONV_CACHE = "general_thai_conversations.json"
NUM_GENERAL_CONVERSATIONS = 1000

# Conversation topic templates — self-contained, no external data needed
TOPIC_TEMPLATES = [
    # Greetings & small talk
    "ทักทายและถามสารทุกข์สุกดิบ",
    #"แนะนำตัวเองสั้นๆ",
    "ถามว่าเป็นใคร ทำอะไรได้บ้าง",
    # Opinions & advice
    "ขอคำแนะนำเรื่องการจัดการเวลา",
    "ถามความคิดเห็นเรื่องการเรียนออนไลน์",
    "ขอคำแนะนำเรื่องการออกกำลังกาย",
    "ถามเรื่องวิธีนอนหลับให้ดีขึ้น",
    "ถามเรื่องการดูแลสุขภาพ",
    "ถามเรื่องการพยาบาลเบื้องต้น",
    "ขอคำแนะนำเรื่องการอ่านหนังสือ",
    "การตอบคำถามเกี่ยวกับการดูแลตัวเองในกรณีน้ำหนักสูงหรือต่ำไป",
    "การตอบคำถามเกี่ยวกับการดูแลตัวเองในกรณีความดันสูงหรือต่ำไป"
    "การตอบคำถามเกี่ยวกับการดูแลตัวเองในกรณีมีไข้",
    "การตอบคำถามเกี่ยวกับการดูแลตัวเองในกรณีผอมหรืออ้วน",
    "ถามวิธีลดความเครียด",
    "ขอคำแนะนำการเริ่มต้นเรียนภาษาใหม่",
    # Knowledge & explanations
    "อธิบายว่า AI ทำงานอย่างไรแบบง่ายๆ",
    #"อธิบายความแตกต่างระหว่าง Python กับ JavaScript",
    "อธิบายว่าอินเทอร์เน็ตทำงานอย่างไร",
    "อธิบายเรื่องพลังงานแสงอาทิตย์",
    "ถามเกี่ยวกับประวัติศาสตร์ไทยสั้นๆ",
    #"อธิบายว่า blockchain คืออะไร",
    "ถามเรื่องระบบสุริยะ",
    # Math & logic
    #"ถามคำถามคณิตศาสตร์ง่ายๆ เช่น บวก ลบ คูณ หาร",
    #"ถามปัญหาคณิตศาสตร์เรื่องเปอร์เซ็นต์",
    #"ถามเรื่องการคำนวณพื้นที่หรือปริมาตร",
    # Language & translation
    #"ถามว่าคำศัพท์ภาษาอังกฤษแปลว่าอะไร",
    #"ขอตัวอย่างประโยคภาษาอังกฤษ",
    #"ถามเรื่องไวยากรณ์ภาษาไทย",
    # Culture & fun
    "ถามเกี่ยวกับเทศกาลสงกรานต์",
    "ถามเกี่ยวกับอาหารไทยชนิดต่างๆ",
    "เล่ามุกตลก",
    "ถามปริศนาง่ายๆ",
    # Daily life
    "ถามขั้นตอนการทำกาแฟ",
    "ถามเรื่องอาหารไทย",
    "ถามวิธีปลูกต้นไม้ในบ้าน",
    "ถามเรื่องการดูแลสัตว์เลี้ยง",
    "ถามเรื่องการทำความสะอาดบ้าน",
    # Thank you & goodbye
    "ขอบคุณและลาก่อน",
    "ถามว่ามีอะไรจะช่วยอีกไหม",
]

print(f"Topic templates: {len(TOPIC_TEMPLATES)}")


In [ ]:
import re

def generate_conversation(topic, conv_id=0):
    """
    Generate a single Thai user-assistant conversation pair.
    Returns: [{"role": "user", "content": ...}, {"role": "assistant", "content": ...}]
    """
    prompt = (
        f"สร้างบทสนทนาภาษาไทย 1 รอบ ระหว่าง user กับ assistant "
        f"ในหัวข้อ: {topic}\n\n"
        f"กฎ:\n"
        f"- ห้ามถามเรื่องที่ต้องเข้าถึงข้อมูลภายนอก เช่น สภาพอากาศ, ร้านอาหารแนะนำ, ข่าว, ราคาหุ้น\n"
        f"- user ถามเป็นภาษาไทยธรรมชาติ 1 ประโยค\n"
        f"- ห้ามตอบว่าคุณมีหน้าที่อะไร เชี่ยวชาญทางด้านอะไร\n"
        f"- assistant ตอบเป็นภาษาไทยธรรมชาติ 1-3 ประโยค\n"
        f"- สร้างบทสนทนาที่แตกต่างจากครั้งก่อน (variation #{conv_id})\n\n"
        f"ตอบเป็น JSON เท่านั้น ห้ามใส่ข้อความอื่นก่อนหรือหลัง:\n"
        f'{{"user": "ข้อความภาษาไทย", "assistant": "ข้อความภาษาไทย"}}'
    )
    for attempt in range(3):  # Retry up to 3 times
        try:
            resp = client.chat.completions.create(
                model=TRANSLATION_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.5,  # Lower temp for reliable JSON (was 0.9)
                max_tokens=300,
            )
            raw = resp.choices[0].message.content.strip()

            # Strategy 1: Extract from code blocks
            if "```" in raw:
                blocks = raw.split("```")
                for block in blocks[1::2]:  # Every other block (inside fences)
                    block = block.strip()
                    if block.startswith("json"):
                        block = block[4:].strip()
                    try:
                        parsed = json.loads(block)
                        if "user" in parsed and "assistant" in parsed:
                            return [
                                {"role": "user", "content": parsed["user"].strip()},
                                {"role": "assistant", "content": parsed["assistant"].strip()},
                            ]
                    except json.JSONDecodeError:
                        continue

            # Strategy 2: Find JSON object with regex
            json_match = re.search(r'\{[^{}]*"user"\s*:\s*"[^"]+"[^{}]*"assistant"\s*:\s*"[^"]+"[^{}]*\}', raw)
            if json_match:
                parsed = json.loads(json_match.group())
                return [
                    {"role": "user", "content": parsed["user"].strip()},
                    {"role": "assistant", "content": parsed["assistant"].strip()},
                ]

            # Strategy 3: Direct JSON parse
            parsed = json.loads(raw)
            if "user" in parsed and "assistant" in parsed:
                return [
                    {"role": "user", "content": parsed["user"].strip()},
                    {"role": "assistant", "content": parsed["assistant"].strip()},
                ]
        except (json.JSONDecodeError, KeyError, Exception):
            continue  # Retry

    return None


def generate_all_conversations(n=NUM_GENERAL_CONVERSATIONS, cache_file=GENERAL_CONV_CACHE):
    """
    Generate n Thai general conversations, cycling through topics.
    """
    if os.path.exists(cache_file):
        print(f"📂 Loading cached conversations from {cache_file}")
        with open(cache_file, "r", encoding="utf-8") as f:
            return json.load(f)

    conversations = []
    failed = 0

    for i in tqdm(range(n), desc="Generating Thai conversations"):
        topic = TOPIC_TEMPLATES[i % len(TOPIC_TEMPLATES)]
        variation_id = i // len(TOPIC_TEMPLATES)

        conv = generate_conversation(topic, conv_id=variation_id)
        if conv:
            conversations.append(conv)
        else:
            failed += 1

        # Checkpoint every 100
        if (i + 1) % 100 == 0:
            with open(cache_file + ".partial", "w", encoding="utf-8") as f:
                json.dump(conversations, f, ensure_ascii=False, indent=2)
            print(f"  💾 Checkpoint ({len(conversations)} done, {failed} failed)")

        time.sleep(0.2)  # Be nice to the endpoint

    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(conversations, f, ensure_ascii=False, indent=2)

    partial = cache_file + ".partial"
    if os.path.exists(partial):
        os.remove(partial)

    print(f"\n✅ Generated {len(conversations)} conversations ({failed} failed)")
    print(f"   Saved to {cache_file}")
    return conversations


general_conversations = generate_all_conversations()


In [ ]:
# Preview generated conversations
print(f"Total general conversations: {len(general_conversations)}")
print()

for i in range(min(5, len(general_conversations))):
    conv = general_conversations[i]
    print(f"--- Conversation {i} ---")
    for turn in conv:
        print(f"  [{turn['role'].upper()}]: {turn['content']}")
    print()


## 4. Translate All Datasets to Thai

**Strategy:**
- Translate only the `user` messages (natural language prompts) to Thai
- Keep `assistant` messages (JSON tool calls) unchanged — function names and arguments are language-agnostic
- Cache translated results to disk (separate caches per dataset)
- **Canfield**: ~1,000 user messages to translate
- **Mobile-actions**: ~9,654 user messages to translate
- **Home Assistant**: ~5,000 user messages to translate


In [ ]:
CACHE_FILE = "translated_thai_dataset.json"


def translate_to_thai(text, max_retries=3):
    """
    Translate English text to Thai via the LM Studio endpoint.
    Includes retry logic for robustness.
    """
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=TRANSLATION_MODEL,
                messages=[
                    {
                        "role": "user",
                        "content": (
                            "Translate the following English text to Thai. "
                            "Output ONLY the Thai translation, nothing else. "
                            "Do not add explanations or notes.\n\n"
                            f"Text: {text}"
                        ),
                    }
                ],
                temperature=0.3,
                max_tokens=512,
            )
            result = response.choices[0].message.content.strip()
            # Basic validation: result should contain Thai characters
            if any('\u0e00' <= c <= '\u0e7f' for c in result):
                return result
            else:
                # If no Thai characters, retry
                if attempt < max_retries - 1:
                    time.sleep(1)
                    continue
                return result  # Return anyway on last attempt
        except Exception as e:
            print(f"  ⚠️ Attempt {attempt + 1} failed for '{text[:50]}...': {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff
            else:
                print(f"  ❌ All retries exhausted. Keeping original text.")
                return text  # Fallback to original


def translate_dataset(dataset, cache_file=CACHE_FILE):
    """
    Translate all user messages in the dataset to Thai.
    Results are cached to disk.
    """
    # Check for cached results
    if os.path.exists(cache_file):
        print(f"📂 Loading cached translations from {cache_file}")
        with open(cache_file, "r", encoding="utf-8") as f:
            return json.load(f)

    translated_examples = []

    for i in tqdm(range(len(dataset)), desc="Translating to Thai"):
        messages_data = dataset[i]["message"]

        # Parse JSON string if needed
        if isinstance(messages_data, str):
            try:
                messages_data = json.loads(messages_data)
            except json.JSONDecodeError:
                print(f"  ⚠️ Skipping example {i}: invalid JSON")
                continue

        translated_turns = []
        for turn in messages_data:
            role = turn.get("role", "")
            content = turn.get("content", "")

            if role == "user":
                # Translate user content to Thai
                thai_content = translate_to_thai(content)
                translated_turns.append({"role": "user", "content": thai_content})
            elif role == "assistant":
                # Keep tool calls as-is (JSON structure)
                translated_turns.append({"role": "assistant", "content": content})
            else:
                # Keep any other roles unchanged
                translated_turns.append({"role": role, "content": content})

        translated_examples.append(translated_turns)

        # Save progress every 100 examples
        if (i + 1) % 100 == 0:
            with open(cache_file + ".partial", "w", encoding="utf-8") as f:
                json.dump(translated_examples, f, ensure_ascii=False, indent=2)
            print(f"  💾 Checkpoint saved ({i + 1}/{len(dataset)})")

    # Save final results
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(translated_examples, f, ensure_ascii=False, indent=2)
    print(f"✅ Translation complete! Saved to {cache_file}")

    # Clean up partial file
    partial = cache_file + ".partial"
    if os.path.exists(partial):
        os.remove(partial)

    return translated_examples

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Translate Canfield dataset
# ═══════════════════════════════════════════════════════════════
CANFIELD_CACHE = "translated_canfield.json"
if os.path.exists(CANFIELD_CACHE):
    raw_canfield = []
translated_canfield = translate_dataset(raw_canfield, cache_file=CANFIELD_CACHE)
print(f"Canfield translated: {len(translated_canfield)} examples")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Translate mobile-actions user messages
# ═══════════════════════════════════════════════════════════════
MOBILE_CACHE = "translated_mobile_actions.json"


def translate_mobile_actions(conversations, cache_file=MOBILE_CACHE):
    """
    Translate mobile-actions user messages to Thai.
    conversations = list of (conversation_turns, tool_info_list)
    """
    if os.path.exists(cache_file):
        print(f"📂 Loading cached translations from {cache_file}")
        with open(cache_file, "r", encoding="utf-8") as f:
            return json.load(f)

    translated = []
    for i in tqdm(range(len(conversations)), desc="Translating mobile-actions"):
        conv, tools = conversations[i]
        translated_turns = []
        for turn in conv:
            if turn["role"] == "user":
                thai = translate_to_thai(turn["content"])
                translated_turns.append({"role": "user", "content": thai})
            else:
                translated_turns.append(turn)
        translated.append(translated_turns)

        if (i + 1) % 200 == 0:
            with open(cache_file + ".partial", "w", encoding="utf-8") as f:
                json.dump(translated, f, ensure_ascii=False, indent=2)
            print(f"  💾 Checkpoint ({i+1}/{len(conversations)})")

    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(translated, f, ensure_ascii=False, indent=2)
    partial = cache_file + ".partial"
    if os.path.exists(partial):
        os.remove(partial)
    print(f"✅ Saved to {cache_file}")
    return translated

if os.path.exists(MOBILE_CACHE):
    mobile_conversations = []

translated_mobile = translate_mobile_actions(mobile_conversations)
print(f"Mobile-actions translated: {len(translated_mobile)} examples")
print(f"\nTotal translated: {len(translated_canfield) + len(translated_mobile)} examples")

# Preview
for label, data in [("Canfield", translated_canfield), ("Mobile-actions", translated_mobile)]:
    print(f"\n{'='*60}")
    print(f"{label} sample:")
    print(f"{'='*60}")
    if data:
        for turn in data[0]:
            print(f"  [{turn['role'].upper()}]: {turn['content'][:150]}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Translate Home Assistant user messages
# ═══════════════════════════════════════════════════════════════
HA_CACHE = "translated_homeassistant.json"


def translate_homeassistant(conversations, cache_file=HA_CACHE):
    """
    Translate Home Assistant user messages to Thai.
    conversations = list of (conversation_turns, tool_info_list)
    """
    if os.path.exists(cache_file):
        print(f"📂 Loading cached translations from {cache_file}")
        with open(cache_file, "r", encoding="utf-8") as f:
            return json.load(f)

    translated = []
    for i in tqdm(range(len(conversations)), desc="Translating Home Assistant"):
        conv, tools = conversations[i]
        translated_turns = []
        for turn in conv:
            if turn["role"] == "user":
                thai = translate_to_thai(turn["content"])
                translated_turns.append({"role": "user", "content": thai})
            else:
                translated_turns.append(turn)
        translated.append(translated_turns)

        if (i + 1) % 200 == 0:
            with open(cache_file + ".partial", "w", encoding="utf-8") as f:
                json.dump(translated, f, ensure_ascii=False, indent=2)
            print(f"  💾 Checkpoint ({i+1}/{len(conversations)})")

    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(translated, f, ensure_ascii=False, indent=2)
    partial = cache_file + ".partial"
    if os.path.exists(partial):
        os.remove(partial)
    print(f"✅ Saved to {cache_file}")
    return translated

if os.path.exists(HA_CACHE):
    ha_conversations = []
translated_homeassistant = translate_homeassistant(ha_conversations)
print(f"Home Assistant translated: {len(translated_homeassistant)} examples")
print(f"\nTotal translated: {len(translated_canfield) + len(translated_mobile) + len(translated_homeassistant)} examples")

# Preview
print(f"\n{'='*60}")
print(f"Home Assistant sample:")
print(f"{'='*60}")
if translated_homeassistant:
    for turn in translated_homeassistant[0]:
        print(f"  [{turn['role'].upper()}]: {turn['content'][:150]}")


## 4.5 Build Tool Registry & Generate Descriptions

Merge tool registries from **both** datasets:
- **Canfield**: Extract tool names/args from assistant responses, generate descriptions via Gemma 3
- **Mobile-actions**: Use the **built-in** tool definitions (names, descriptions, parameters) — no generation needed


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Step 1: Extract tool registry from ALL datasets
# ═══════════════════════════════════════════════════════════════
import random

tool_registry = {}  # {tool_name: set_of_arg_keys}

# --- From Canfield ---
for i in range(len(raw_canfield)):
    messages_data = raw_canfield[i]["messages"]
    if isinstance(messages_data, str):
        try:
            messages_data = json.loads(messages_data)
        except json.JSONDecodeError:
            continue
    for turn in messages_data:
        if turn.get("role") == "assistant":
            content = turn.get("content", "")
            try:
                tool_call = json.loads(content)
                name = tool_call.get("name", "")
                args = tool_call.get("arguments", {})
                if name:
                    if name not in tool_registry:
                        tool_registry[name] = set()
                    tool_registry[name].update(args.keys())
            except (json.JSONDecodeError, TypeError, AttributeError):
                continue

canfield_tool_count = len(tool_registry)
print(f"Canfield tools: {canfield_tool_count}")

# --- From mobile-actions (tools field) ---
mobile_tool_descriptions = {}  # {name: {args, description}}
for conv, tools in mobile_conversations:
    for t in tools:
        name = t["name"]
        if name not in tool_registry:
            tool_registry[name] = set()
        tool_registry[name].update(t["args"])
        if name not in mobile_tool_descriptions:
            mobile_tool_descriptions[name] = {
                "args": t["args"],
                "description": t["description"],
            }

mobile_tool_count = len(tool_registry) - canfield_tool_count
print(f"Mobile-actions new tools: {mobile_tool_count}")

# --- From Home Assistant (tools field) ---
ha_tool_descriptions = {}  # {name: {args, description}}
for conv, tools in ha_conversations:
    for t in tools:
        name = t["name"]
        if name not in tool_registry:
            tool_registry[name] = set()
        tool_registry[name].update(t["args"])
        if name not in ha_tool_descriptions:
            ha_tool_descriptions[name] = {
                "args": t["args"],
                "description": t["description"],
            }

ha_tool_count = len(tool_registry) - canfield_tool_count - mobile_tool_count
print(f"Home Assistant new tools: {ha_tool_count}")
print(f"Total unique tools: {len(tool_registry)}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Step 2: Generate/collect tool descriptions
#   - Mobile-actions: use BUILT-IN descriptions (no LLM needed)
#   - Home Assistant: use BUILT-IN descriptions (no LLM needed)
#   - Canfield: generate via Gemma 3 (as before)
# ═══════════════════════════════════════════════════════════════
TOOL_DESC_CACHE = "tool_descriptions_cache.json"


def generate_tool_description(tool_name, arg_keys):
    """Ask Gemma 3 to produce a one-line description for a tool."""
    args_str = ", ".join(sorted(arg_keys))
    prompt = (
        f"Given a function named `{tool_name}` with parameters "
        f"`{args_str}`, write exactly ONE short sentence describing "
        f"what this function does. Output ONLY the description, "
        f"nothing else. Do not include the function name in the description."
    )
    try:
        resp = client.chat.completions.create(
            model=TRANSLATION_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=100,
        )
        desc = resp.choices[0].message.content.strip().strip('"\' ')
        return desc
    except Exception as e:
        print(f"  ⚠️ Failed for {tool_name}: {e}")
        return f"Perform {tool_name.replace('_', ' ')} operation"


if os.path.exists(TOOL_DESC_CACHE):
    print(f"📂 Loading cached tool descriptions from {TOOL_DESC_CACHE}")
    with open(TOOL_DESC_CACHE, "r", encoding="utf-8") as f:
        tool_descriptions = json.load(f)
else:
    tool_descriptions = {}

# Add mobile-actions tools (built-in descriptions, no LLM call)
added_mobile = 0
for name, info in mobile_tool_descriptions.items():
    if name not in tool_descriptions:
        tool_descriptions[name] = {
            "args": sorted(info["args"]),
            "description": info["description"],
        }
        added_mobile += 1
print(f"Added {added_mobile} mobile-actions tools (built-in descriptions)")

# Add Home Assistant tools (built-in descriptions, no LLM call)
added_ha = 0
for name, info in ha_tool_descriptions.items():
    if name not in tool_descriptions:
        tool_descriptions[name] = {
            "args": sorted(info["args"]),
            "description": info["description"],
        }
        added_ha += 1
print(f"Added {added_ha} Home Assistant tools (built-in descriptions)")

# Generate descriptions for Canfield-only tools missing from cache
canfield_only = [n for n in tool_registry if n not in tool_descriptions]
if canfield_only:
    print(f"Generating descriptions for {len(canfield_only)} Canfield tools...")
    for name in tqdm(canfield_only, desc="Generating descriptions"):
        desc = generate_tool_description(name, tool_registry[name])
        tool_descriptions[name] = {
            "args": sorted(tool_registry[name]),
            "description": desc,
        }
        time.sleep(0.3)

# Save updated cache
with open(TOOL_DESC_CACHE, "w", encoding="utf-8") as f:
    json.dump(tool_descriptions, f, ensure_ascii=False, indent=2)
print(f"\n✅ Total tool descriptions: {len(tool_descriptions)}")

# Show a few
for name in list(sorted(tool_descriptions.keys()))[:5]:
    info = tool_descriptions[name]
    print(f"  {name}({', '.join(info['args'])}): {info['description'][:80]}")
if len(tool_descriptions) > 5:
    print(f"  ... and {len(tool_descriptions)-5} more")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Step 3: Helper to build per-example tool lists
# ═══════════════════════════════════════════════════════════════
ALL_TOOL_NAMES = sorted(tool_descriptions.keys())


def format_tool_list(tool_names):
    """
    Format a list of tool names into a human-readable tool list string.
    
    Example output:
      Available tools:
      - filesystem_ops(operation, path, destination): Perform file system operations
      - send_email(to, subject, body): Send an email message
      - weather_check(location): Check weather conditions
    """
    lines = ["Available tools:"]
    for name in tool_names:
        info = tool_descriptions[name]
        args_str = ", ".join(info["args"])
        lines.append(f"- {name}({args_str}): {info['description']}")
    return "\n".join(lines)


def get_correct_tool_name(conversation):
    """Extract the tool name from the assistant's response in a conversation."""
    for turn in conversation:
        if turn.get("role") == "assistant":
            content = turn.get("content", "")
            try:
                tool_call = json.loads(content)
                return tool_call.get("name", None)
            except (json.JSONDecodeError, TypeError):
                continue
    return None


def pick_tools_for_example(correct_tool_name, n_distractors=2):
    """
    Pick the correct tool + n_distractors random other tools.
    Returns a shuffled list of tool names.
    """
    if correct_tool_name is None or correct_tool_name not in tool_descriptions:
        # Fallback: just pick 3 random tools
        return random.sample(ALL_TOOL_NAMES, min(3, len(ALL_TOOL_NAMES)))

    # Pick distractors (different from the correct tool)
    other_tools = [t for t in ALL_TOOL_NAMES if t != correct_tool_name]
    distractors = random.sample(other_tools, min(n_distractors, len(other_tools)))

    # Combine and shuffle
    tool_list = [correct_tool_name] + distractors
    random.shuffle(tool_list)
    return tool_list


# Quick test
'''
if len(translated_data) > 0:
    test_correct = get_correct_tool_name(raw_dataset[0]["message"] if isinstance(raw_dataset[0]["message"], list) else json.loads(raw_dataset[0]["message"]))
    test_tools = pick_tools_for_example(test_correct)
    print(f"Correct tool: {test_correct}")
    print(f"Selected tools: {test_tools}")
    print()
    print(format_tool_list(test_tools))'''


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Step 4: Pre-compute tool assignments for ALL datasets
# ═══════════════════════════════════════════════════════════════
random.seed(42)

# Canfield: pick 1 correct + 2 random distractors per example
canfield_tool_assignments = []
for i in range(len(raw_canfield)):
    msgs = raw_canfield[i]["message"]
    if isinstance(msgs, str):
        try:
            msgs = json.loads(msgs)
        except json.JSONDecodeError:
            canfield_tool_assignments.append(random.sample(ALL_TOOL_NAMES, min(3, len(ALL_TOOL_NAMES))))
            continue
    correct_tool = get_correct_tool_name(msgs)
    assigned = pick_tools_for_example(correct_tool, n_distractors=2)
    canfield_tool_assignments.append(assigned)

print(f"✅ Canfield tool assignments: {len(canfield_tool_assignments)}")

# Mobile-actions: use the REAL tool list from the dataset
mobile_tool_assignments = []
for conv, tools in mobile_conversations:
    tool_names = [t["name"] for t in tools]
    mobile_tool_assignments.append(tool_names)

print(f"✅ Mobile-actions tool assignments: {len(mobile_tool_assignments)}")
print(f"   (using built-in tool lists from the dataset)")

# Home Assistant: use the REAL tool list from the dataset
ha_tool_assignments = []
for conv, tools in ha_conversations:
    tool_names = [t["name"] for t in tools]
    ha_tool_assignments.append(tool_names)

print(f"✅ Home Assistant tool assignments: {len(ha_tool_assignments)}")
print(f"   (using built-in tool lists from the dataset)")

# Show samples
print(f"\nCanfield sample: {canfield_tool_assignments[0]}")
print(f"Mobile sample:   {mobile_tool_assignments[0]}")
print(f"HA sample:       {ha_tool_assignments[0][:5]}{'...' if len(ha_tool_assignments[0]) > 5 else ''}")


## 6. Format Data — JSON Output (Matches Server Template)



In [ ]:
# Load the FunctionGemma processor/tokenizer
MODEL_ID = "google/functiongemma-270m-it"

processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor  # AutoProcessor wraps the tokenizer

print(f"Tokenizer loaded: {type(processor)}")
print(f"Vocab size: {processor.tokenizer.vocab_size if hasattr(processor, 'tokenizer') else 'N/A'}")

# ═══════════════════════════════════════════════════════════════
# Define CUSTOM TOOL SCHEMAS (OpenAI format — same as LM Studio API)
# These will be used for both training data generation AND formatting.
# ═══════════════════════════════════════════════════════════════
CUSTOM_TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather conditions for a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "The city name, e.g. Bangkok"},
                    "format": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "The temperature unit to use",
                    },
                },
                "required": ["location", "format"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_location",
            "description": "Search for places, businesses, or points of interest near a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query, e.g. coffee shop"},
                    "location": {"type": "string", "description": "The city or area to search in"},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_forecast",
            "description": "Get weather forecast for upcoming days at a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "The city name"},
                    "days": {"type": "integer", "description": "Number of forecast days (1-7)"},
                },
                "required": ["location"],
            },
        },
    },
]

print(f"Custom tool schemas: {len(CUSTOM_TOOL_SCHEMAS)}")
for t in CUSTOM_TOOL_SCHEMAS:
    print(f"  - {t['function']['name']}: {t['function']['description']}")


In [ ]:
def tool_call_to_json_format(tool_call_json_str):
    """
    Convert raw tool call JSON to the server's expected format.
    Input:  {"name": "func", "arguments": {"key": "val"}}
    Output: {"tool_call": {"name": "func", "arguments": {"key": "val"}}}
    """
    try:
        tool_call = json.loads(tool_call_json_str)
    except (json.JSONDecodeError, TypeError):
        # If not valid JSON, wrap as response
        return json.dumps({"response": tool_call_json_str}, ensure_ascii=False)
    
    func_name = tool_call.get("name", "unknown")
    arguments = tool_call.get("arguments", {})
    
    result = {
        "tool_call": {
            "name": func_name,
            "arguments": arguments
        }
    }
    return json.dumps(result, ensure_ascii=False)


def text_to_json_response(text):
    """
    Wrap plain text in the server's response format.
    Input:  "สวัสดีครับ"
    Output: {"response": "สวัสดีครับ"}
    """
    return json.dumps({"response": text}, ensure_ascii=False)


def build_tool_schemas_from_names(tool_names):
    """
    Convert tool names + descriptions into OpenAI-format schemas
    so we can use processor.apply_chat_template(tools=...).
    """
    schemas = []
    for name in tool_names:
        if name not in tool_descriptions:
            continue
        info = tool_descriptions[name]
        props = {}
        for arg in info["args"]:
            props[arg] = {"type": "string", "description": arg}
        schemas.append({
            "type": "function",
            "function": {
                "name": name,
                "description": info["description"],
                "parameters": {
                    "type": "object",
                    "properties": props,
                    "required": list(props.keys()),
                },
            },
        })
    return schemas


# Test
test_json = '{"name": "get_current_weather", "arguments": {"location": "Bangkok", "format": "celsius"}}'
print("Tool call JSON format:")
print(f"  {tool_call_to_json_format(test_json)}")
print("Response JSON format:")
print(f"  {text_to_json_response('สวัสดีครับ ยินดีช่วยเหลือค่ะ')}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# v9: NO manual developer message!
# Let processor.apply_chat_template(tools=...) generate the
# developer prompt automatically — matches what servers send.
#
# Output format is JSON:
#   Tool call:  {"tool_call": {"name": "func", "arguments": {...}}}
#   Response:   {"response": "Thai text"}
# ═══════════════════════════════════════════════════════════════

def build_training_text(translated_conversation, tool_schemas=None, is_tool_call=True):
    """
    Build training text using processor.apply_chat_template(tools=...).
    No manual developer message — the template adds it automatically.

    Args:
        translated_conversation: [{role, content}, ...]
        tool_schemas: list of OpenAI-format tool schemas, or None
        is_tool_call: if True, assistant content is a tool call JSON;
                      if False, assistant content is plain text (wrap in {"response": ...})
    """
    formatted_messages = []
    # NO developer message here — template generates it!

    for turn in translated_conversation:
        role = turn["role"]
        content = turn["content"]
        if role == "user":
            formatted_messages.append({"role": "user", "content": content})
        elif role == "assistant":
            if is_tool_call:
                # Convert to {"tool_call": {"name": ..., "arguments": ...}}
                json_output = tool_call_to_json_format(content)
            else:
                # Wrap plain text in {"response": "..."}
                json_output = text_to_json_response(content)
            formatted_messages.append({"role": "model", "content": json_output})
        else:
            formatted_messages.append({"role": role, "content": content})

    try:
        if tool_schemas:
            text = processor.apply_chat_template(
                formatted_messages,
                tools=tool_schemas,
                tokenize=False,
                add_generation_prompt=False,
            )
        else:
            # No tools — still use template for consistent format
            text = processor.apply_chat_template(
                formatted_messages,
                tokenize=False,
                add_generation_prompt=False,
            )
        return text
    except Exception as e:
        print(f"Error applying chat template: {e}")
        return None


# ═══════════════════════════════════════════════════════════════
# Test: verify the training format matches server format
# ═══════════════════════════════════════════════════════════════
test_tool_call = [
    {"role": "user", "content": "วันนี้อากาศเป็นยังไงที่กรุงเทพ"},
    {"role": "assistant", "content": '{"name": "get_current_weather", "arguments": {"location": "Bangkok", "format": "celsius"}}'},
]
test_chat = [
    {"role": "user", "content": "สวัสดีครับ"},
    {"role": "assistant", "content": "สวัสดีค่ะ ยินดีช่วยเหลือค่ะ"},
]

print("=" * 60)
print("TOOL CALL example (should have tool_call JSON):")
print("=" * 60)
sample1 = build_training_text(test_tool_call, tool_schemas=CUSTOM_TOOL_SCHEMAS, is_tool_call=True)
print(sample1)

print("\n" + "=" * 60)
print("CHAT example with tools (should have response JSON):")
print("=" * 60)
sample2 = build_training_text(test_chat, tool_schemas=CUSTOM_TOOL_SCHEMAS, is_tool_call=False)
print(sample2)

print("\n" + "=" * 60)
print("CHAT example without tools:")
print("=" * 60)
sample3 = build_training_text(test_chat, tool_schemas=None, is_tool_call=False)
print(sample3)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Build COMBINED training dataset — v9: JSON output format
# Tool calls:  {"tool_call": {"name": ..., "arguments": ...}}
# Responses:   {"response": "Thai text"}
# ═══════════════════════════════════════════════════════════════
training_texts = []
skipped = 0

def names_to_schemas(tool_names):
    return build_tool_schemas_from_names(tool_names) if tool_names else None

# --- Canfield examples (tool calls) ---
for i, conv in enumerate(tqdm(translated_canfield, desc="Formatting Canfield")):
    tools = canfield_tool_assignments[i] if i < len(canfield_tool_assignments) else None
    schemas = names_to_schemas(tools)
    text = build_training_text(conv, tool_schemas=schemas, is_tool_call=True)
    if text and len(text.strip()) > 10:
        training_texts.append({"text": text, "source": "canfield"})
    else:
        skipped += 1

canfield_count = len(training_texts)
print(f"  Canfield: {canfield_count} examples")

# --- Mobile-actions examples (tool calls) ---
for i, conv in enumerate(tqdm(translated_mobile, desc="Formatting mobile-actions")):
    tools = mobile_tool_assignments[i] if i < len(mobile_tool_assignments) else None
    schemas = names_to_schemas(tools)
    text = build_training_text(conv, tool_schemas=schemas, is_tool_call=True)
    if text and len(text.strip()) > 10:
        training_texts.append({"text": text, "source": "mobile"})
    else:
        skipped += 1

mobile_count = len(training_texts) - canfield_count
print(f"  Mobile-actions: {mobile_count} examples")

# --- Home Assistant examples (tool calls) ---
for i, conv in enumerate(tqdm(translated_homeassistant, desc="Formatting Home Assistant")):
    tools = ha_tool_assignments[i] if i < len(ha_tool_assignments) else None
    schemas = names_to_schemas(tools)
    text = build_training_text(conv, tool_schemas=schemas, is_tool_call=True)
    if text and len(text.strip()) > 10:
        training_texts.append({"text": text, "source": "homeassistant"})
    else:
        skipped += 1

ha_count = len(training_texts) - canfield_count - mobile_count
print(f"  Home Assistant: {ha_count} examples")

# --- Custom tool examples (weather, location, forecast) ---
CUSTOM_EXAMPLES = [
    # get_current_weather
    ({"role": "user", "content": "วันนี้อากาศเป็นยังไงที่กรุงเทพ"}, {"role": "assistant", "content": '{"name": "get_current_weather", "arguments": {"location": "Bangkok", "format": "celsius"}}'}),
    ({"role": "user", "content": "อากาศที่เชียงใหม่เป็นยังไงบ้าง"}, {"role": "assistant", "content": '{"name": "get_current_weather", "arguments": {"location": "Chiang Mai", "format": "celsius"}}'}),
    ({"role": "user", "content": "ตอนนี้หนาวไหมที่ภูเก็ต"}, {"role": "assistant", "content": '{"name": "get_current_weather", "arguments": {"location": "Phuket", "format": "celsius"}}'}),
    ({"role": "user", "content": "เช็คสภาพอากาศที่ขอนแก่น"}, {"role": "assistant", "content": '{"name": "get_current_weather", "arguments": {"location": "Khon Kaen", "format": "celsius"}}'}),
    ({"role": "user", "content": "อุณหภูมิที่นครราชสีมาเท่าไหร่"}, {"role": "assistant", "content": '{"name": "get_current_weather", "arguments": {"location": "Nakhon Ratchasima", "format": "celsius"}}'}),
    ({"role": "user", "content": "ฝนตกไหมที่หาดใหญ่"}, {"role": "assistant", "content": '{"name": "get_current_weather", "arguments": {"location": "Hat Yai", "format": "celsius"}}'}),
    ({"role": "user", "content": "อากาศที่ลอนดอนเป็นยังไง"}, {"role": "assistant", "content": '{"name": "get_current_weather", "arguments": {"location": "London", "format": "celsius"}}'}),
    ({"role": "user", "content": "อุณหภูมิที่นิวยอร์กกี่องศาฟาเรนไฮต์"}, {"role": "assistant", "content": '{"name": "get_current_weather", "arguments": {"location": "New York", "format": "fahrenheit"}}'}),
    ({"role": "user", "content": "ร้อนไหมวันนี้ที่อุดรธานี"}, {"role": "assistant", "content": '{"name": "get_current_weather", "arguments": {"location": "Udon Thani", "format": "celsius"}}'}),
    # search_location
    ({"role": "user", "content": "หาร้านกาแฟใกล้ๆ หน่อย"}, {"role": "assistant", "content": '{"name": "search_location", "arguments": {"query": "coffee shop"}}'}),
    ({"role": "user", "content": "ร้านอาหารไทยแถวสยามมีที่ไหนบ้าง"}, {"role": "assistant", "content": '{"name": "search_location", "arguments": {"query": "Thai restaurant", "location": "Siam"}}'}),
    ({"role": "user", "content": "หาปั๊มน้ำมันใกล้เชียงใหม่"}, {"role": "assistant", "content": '{"name": "search_location", "arguments": {"query": "gas station", "location": "Chiang Mai"}}'}),
    ({"role": "user", "content": "โรงพยาบาลใกล้ที่สุดอยู่ตรงไหน"}, {"role": "assistant", "content": '{"name": "search_location", "arguments": {"query": "hospital"}}'}),
    ({"role": "user", "content": "ATM ใกล้ๆ มีที่ไหนบ้าง"}, {"role": "assistant", "content": '{"name": "search_location", "arguments": {"query": "ATM"}}'}),
    ({"role": "user", "content": "ร้านสะดวกซื้อแถวนี้"}, {"role": "assistant", "content": '{"name": "search_location", "arguments": {"query": "convenience store"}}'}),
    ({"role": "user", "content": "หาร้านซ่อมมือถือที่ MBK"}, {"role": "assistant", "content": '{"name": "search_location", "arguments": {"query": "phone repair", "location": "MBK"}}'}),
    ({"role": "user", "content": "ร้านค้าใกล้สนามบินสุวรรณภูมิ"}, {"role": "assistant", "content": '{"name": "search_location", "arguments": {"query": "shop", "location": "Suvarnabhumi Airport"}}'}),
    # get_forecast
    ({"role": "user", "content": "พยากรณ์อากาศ 3 วันข้างหน้าที่กรุงเทพ"}, {"role": "assistant", "content": '{"name": "get_forecast", "arguments": {"location": "Bangkok", "days": 3}}'}),
    ({"role": "user", "content": "สัปดาห์หน้าอากาศเป็นยังไงที่เชียงราย"}, {"role": "assistant", "content": '{"name": "get_forecast", "arguments": {"location": "Chiang Rai", "days": 7}}'}),
    ({"role": "user", "content": "พรุ่งนี้ฝนตกไหมที่กรุงเทพ"}, {"role": "assistant", "content": '{"name": "get_forecast", "arguments": {"location": "Bangkok", "days": 1}}'}),
    ({"role": "user", "content": "อากาศ 5 วันข้างหน้าที่พัทยา"}, {"role": "assistant", "content": '{"name": "get_forecast", "arguments": {"location": "Pattaya", "days": 5}}'}),
]

custom_count = 0
for user_turn, asst_turn in CUSTOM_EXAMPLES:
    conv = [user_turn, asst_turn]
    text = build_training_text(conv, tool_schemas=CUSTOM_TOOL_SCHEMAS, is_tool_call=True)
    if text and len(text.strip()) > 10:
        training_texts.append({"text": text, "source": "custom"})
        custom_count += 1

# Duplicate custom examples 8x
custom_base = [t for t in training_texts if t["source"] == "custom"]
for _ in range(7):
    training_texts.extend([dict(t) for t in custom_base])
    custom_count += len(custom_base)

print(f"  Custom tools: {custom_count} examples (with 8x duplication)")

# --- General conversations (response JSON) ---
# v9: output is {"response": "Thai text"}
random.seed(123)
general_with_tools = 0
general_no_tools = 0
for i, conv in enumerate(tqdm(general_conversations, desc="Formatting general conversations")):
    if i % 2 == 0:
        # WITH tools — model learns to respond (not call tool) when chatting
        n_tools = random.randint(2, min(5, len(ALL_TOOL_NAMES)))
        random_tools = random.sample(ALL_TOOL_NAMES, n_tools)
        schemas = names_to_schemas(random_tools)
        text = build_training_text(conv, tool_schemas=schemas, is_tool_call=False)
        general_with_tools += 1
    else:
        # WITHOUT tools
        text = build_training_text(conv, tool_schemas=None, is_tool_call=False)
        general_no_tools += 1
    if text and len(text.strip()) > 10:
        training_texts.append({"text": text, "source": "general"})
    else:
        skipped += 1

general_count = general_with_tools + general_no_tools
print(f"  General: {general_count} ({general_with_tools} with tools, {general_no_tools} without)")

# Duplicate general conversations 3x (boost from ~414 to ~1242)
general_base = [t for t in training_texts if t["source"] == "general"]
for _ in range(2):
    training_texts.extend([dict(t) for t in general_base])
    general_count += len(general_base)
print(f"  General (after 3x duplication): {general_count}")

# --- Tool-rejection examples (response JSON with refusal) ---
REJECTION_RATIO = 0.02
REFUSAL_RESPONSES_TH = [
    "ขออภัย ฉันไม่มีเครื่องมือที่เหมาะสมสำหรับคำขอนี้",
    "ไม่สามารถดำเนินการได้ เนื่องจากไม่มีเครื่องมือที่ตรงกับคำขอ",
    "ขออภัย ไม่มีฟังก์ชันที่รองรับคำสั่งนี้ในรายการเครื่องมือที่มี",
]

rejection_count = 0
random.seed(456)
tool_call_sources = []
for i, conv in enumerate(translated_canfield):
    ct = canfield_tool_assignments[i] if i < len(canfield_tool_assignments) else []
    if ct: tool_call_sources.append((conv, ct))
for i, conv in enumerate(translated_mobile):
    ct = mobile_tool_assignments[i] if i < len(mobile_tool_assignments) else []
    if ct: tool_call_sources.append((conv, ct))
for i, conv in enumerate(translated_homeassistant):
    ct = ha_tool_assignments[i] if i < len(ha_tool_assignments) else []
    if ct: tool_call_sources.append((conv, ct))

n_rejections = int(len(tool_call_sources) * REJECTION_RATIO)
rejection_indices = random.sample(range(len(tool_call_sources)), min(n_rejections, len(tool_call_sources)))

for idx in tqdm(rejection_indices, desc="Building rejection examples"):
    conv, correct_tools = tool_call_sources[idx]
    wrong_tools = [t for t in ALL_TOOL_NAMES if t not in correct_tools]
    if len(wrong_tools) < 2: continue
    n_wrong = random.randint(2, min(4, len(wrong_tools)))
    distractor_tools = random.sample(wrong_tools, n_wrong)
    schemas = names_to_schemas(distractor_tools)
    rejection_conv = [
        {"role": "user", "content": conv[0]["content"]},
        {"role": "assistant", "content": random.choice(REFUSAL_RESPONSES_TH)},
    ]
    # is_tool_call=False → wraps refusal as {"response": "ขออภัย..."}
    text = build_training_text(rejection_conv, tool_schemas=schemas, is_tool_call=False)
    if text and len(text.strip()) > 10:
        training_texts.append({"text": text, "source": "rejection"})
        rejection_count += 1

print(f"  Tool-rejection: {rejection_count} examples")

total = len(training_texts)
print(f"\n✅ Combined: {total} training examples ({skipped} skipped)")
print(f"   Canfield:       {canfield_count:>5} ({canfield_count/total*100:.1f}%)")
print(f"   Mobile:         {mobile_count:>5} ({mobile_count/total*100:.1f}%)")
print(f"   Home Assistant: {ha_count:>5} ({ha_count/total*100:.1f}%)")
print(f"   Custom tools:   {custom_count:>5} ({custom_count/total*100:.1f}%)")
print(f"   General:        {general_count:>5} ({general_count/total*100:.1f}%)")
print(f"   Rejection:      {rejection_count:>5} ({rejection_count/total*100:.1f}%)")

import random as _rng
_rng.seed(42)
_rng.shuffle(training_texts)

train_dataset = Dataset.from_list([{"text": t["text"]} for t in training_texts])
print(f"\nDataset columns: {train_dataset.column_names}")
print(f"Sample text length: {len(train_dataset[0]['text'])} chars")


In [ ]:
# Split into train/eval (90/10)
split = train_dataset.train_test_split(test_size=0.1, seed=42)
train_split = split["train"]
eval_split = split["test"]

print(f"Train set: {len(train_split)} examples")
print(f"Eval set:  {len(eval_split)} examples")

# Show a few training examples
for i in range(min(2, len(train_split))):
    print(f"\n{'─'*60}")
    print(f"Training example {i}:")
    print(f"{'─'*60}")
    print(train_split[i]["text"][:500])

## 7. Fine-tune with LoRA + SFTTrainer

We use LoRA (Low-Rank Adaptation) for parameter-efficient fine-tuning.

**Platform notes:**
- **CUDA (RTX 3090)**: Uses bf16 mixed-precision — fastest (~5-10 min)
- **Apple Silicon (M1–M4 MPS)**: Uses float32 for stability — fits easily in unified memory (~15-30 min)
- **CPU**: Uses float32 (slowest, but works)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Select appropriate dtype for the detected device
# ═══════════════════════════════════════════════════════════════
if DEVICE == 'cuda':
    model_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif DEVICE == 'mps':
    # MPS: float32 is the most stable for training
    # The 270M model is only ~1GB in float32 — fits easily in M4's unified memory
    model_dtype = torch.float32
else:
    model_dtype = torch.float32

print(f"Using dtype: {model_dtype} on device: {DEVICE}")

# Load base model
if DEVICE == 'cuda':
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        torch_dtype=model_dtype,
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=model_dtype,
    )
    base_model = base_model.to(DEVICE)

# Load tokenizer for training
train_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
train_tokenizer.padding_side = "right"

# Ensure pad token is set
if train_tokenizer.pad_token is None:
    train_tokenizer.pad_token = train_tokenizer.eos_token
    base_model.config.pad_token_id = train_tokenizer.eos_token_id

# ═══════════════════════════════════════════════════════════════
# FIX: Resize model embeddings to match tokenizer vocab size
# This prevents vocab size mismatch during GGUF conversion.
# If the tokenizer has extra special tokens (e.g. from chat template),
# the embedding matrix must be resized to match.
# ═══════════════════════════════════════════════════════════════
orig_vocab = base_model.config.vocab_size
tokenizer_vocab = len(train_tokenizer)
if orig_vocab != tokenizer_vocab:
    print(f"⚠️ Vocab mismatch: model={orig_vocab}, tokenizer={tokenizer_vocab}")
    print(f"   Resizing model embeddings to {tokenizer_vocab}...")
    base_model.resize_token_embeddings(tokenizer_vocab)
    print(f"   ✅ Resized successfully")
else:
    print(f"✅ Vocab size matches: {orig_vocab}")

print(f"Model loaded on: {next(base_model.parameters()).device}")
print(f"Model dtype: {model_dtype}")
print(f"Total parameters: {sum(p.numel() for p in base_model.parameters()) / 1e6:.1f}M")


In [ ]:
# LoRA Configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Apply LoRA
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig
# Output directory
OUTPUT_DIR = "./functiongemma-thai-toolcall"

# ═══════════════════════════════════════════════════════════════
# Configure training precision based on device
# ═══════════════════════════════════════════════════════════════
use_fp16 = False
use_bf16 = False

if DEVICE == 'cuda':
    if torch.cuda.is_bf16_supported():
        use_bf16 = True
    else:
        use_fp16 = True
elif DEVICE == 'mps':
    # MPS: keep fp32 for training stability
    # fp16/bf16 AMP is not fully supported on MPS yet
    use_fp16 = False
    use_bf16 = False

print(f"Training precision — fp16: {use_fp16}, bf16: {use_bf16}")

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=use_fp16,
    bf16=use_bf16,
    optim="adamw_torch",
    report_to="none",
    push_to_hub=False,
    max_length=1024,
    completion_only_loss=True

    # Enable MPS backend for Apple Silicon
    #use_mps_device=(DEVICE == 'mps'),
)
base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=model_dtype,
        token=HF_TOKEN,
    )
base_model.config.pad_token_id = train_tokenizer.pad_token_id  # 0
base_model.config.eos_token_id = train_tokenizer.eos_token_id  # 1
base_model.config.bos_token_id = train_tokenizer.bos_token_id
# SFTTrainer
trainer = SFTTrainer(
    model=base_model,
    args=training_args,
    train_dataset=train_split,
    eval_dataset=eval_split,
    processing_class=train_tokenizer,
    peft_config=lora_config,
)

print("\n🚀 Starting training...")

In [ ]:
# Train!
train_result = trainer.train()

# Print training metrics
print("\n" + "=" * 40)
print("Training Complete!")
print("=" * 40)
print(f"Training loss: {train_result.training_loss:.4f}")
metrics = train_result.metrics
for key, value in metrics.items():
    print(f"  {key}: {value}")

In [ ]:
# Plot training loss
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_losses = [log["loss"] for log in log_history if "loss" in log]
train_steps = [log["step"] for log in log_history if "loss" in log]
eval_losses = [log["eval_loss"] for log in log_history if "eval_loss" in log]
eval_steps = [log["step"] for log in log_history if "eval_loss" in log]

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_losses, label="Training Loss", alpha=0.8)
if eval_losses:
    plt.plot(eval_steps, eval_losses, label="Validation Loss", alpha=0.8, marker="o")
plt.xlabel("Steps")
plt.ylabel("Loss")
plt.title("FunctionGemma Thai Fine-tuning: Training & Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Merge LoRA Adapter & Save Full Model

In [ ]:
# Save the LoRA adapter first
ADAPTER_DIR = "./functiongemma-thai-adapter"
trainer.save_model(ADAPTER_DIR)
train_tokenizer.save_pretrained(ADAPTER_DIR)
print(f"✅ LoRA adapter saved to {ADAPTER_DIR}")

# Merge adapter into base model for deployment
MERGED_DIR = "./functiongemma-thai-merged"

# Reload base model on CPU for merging (works on all platforms)
merge_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="cpu",
    torch_dtype=model_dtype,
)

# FIX: Resize embeddings to match tokenizer (same fix as during training)
merge_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
if merge_base.config.vocab_size != len(merge_tokenizer):
    print(f"Resizing merge model embeddings: {merge_base.config.vocab_size} → {len(merge_tokenizer)}")
    merge_base.resize_token_embeddings(len(merge_tokenizer))

# Load and merge LoRA adapter
merged_model = PeftModel.from_pretrained(merge_base, ADAPTER_DIR)
merged_model = merged_model.merge_and_unload()

# Save merged model
merged_model.save_pretrained(MERGED_DIR)
merge_tokenizer.save_pretrained(MERGED_DIR)

print(f"✅ Merged model saved to {MERGED_DIR}")
print(f"   Model vocab: {merged_model.config.vocab_size}")
print(f"   Tokenizer vocab: {len(merge_tokenizer)}")
print(f"   Files: {os.listdir(MERGED_DIR)}")


## 9. Convert to GGUF for Ollama (Raspberry Pi 5)

To run on Raspberry Pi 5 with Ollama, we need to convert the model to GGUF format.

We use **Q4_K_M** quantization for a good balance between quality and size (~150MB for 270M params).

In [ ]:
# Clone llama.cpp for GGUF conversion
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git llama_cpp_convert 2>/dev/null || echo "Already cloned"
!pip install -q -r llama_cpp_convert/requirements/requirements-convert_hf_to_gguf.txt 2>/dev/null || true

In [ ]:
GGUF_OUTPUT = "./functiongemma-thai-toolcall-fp16.gguf"

# Step 1: Convert HF model to GGUF (F16)
!python llama_cpp_convert/convert_hf_to_gguf.py \
    {MERGED_DIR} \
    --outfile {GGUF_OUTPUT} \
    --outtype f16

if os.path.exists(GGUF_OUTPUT):
    size_mb = os.path.getsize(GGUF_OUTPUT) / (1024 * 1024)
    print(f"\n✅ GGUF model created: {GGUF_OUTPUT} ({size_mb:.1f} MB)")
else:
    print("\n❌ GGUF conversion failed. See errors above.")

In [ ]:
# Step 2: Quantize to Q4_K_M for Raspberry Pi 5
QuantatizeMethod = 'Q8_0'
# This requires llama-quantize binary. Build it if not available:
QUANTIZED_GGUF = f"./functiongemma-thai-toolcall-{QuantatizeMethod}.gguf"

# Try using the pre-built quantize tool, or build from source
import shutil
import platform

quantize_bin = shutil.which("llama-quantize")

if quantize_bin is None:
    print("Building llama-quantize from source...")
    # Use sysctl on macOS, nproc on Linux
    ncpu_cmd = "sysctl -n hw.ncpu" if platform.system() == "Darwin" else "nproc"
    !cd llama_cpp_convert && cmake -B build && cmake --build build --target llama-quantize -j$({ncpu_cmd}) 2>&1 | tail -5
    quantize_bin = "llama_cpp_convert/build/bin/llama-quantize"

if os.path.exists(str(quantize_bin)):
    !{quantize_bin} {GGUF_OUTPUT} {QUANTIZED_GGUF} {QuantatizeMethod}
    if os.path.exists(QUANTIZED_GGUF):
        size_mb = os.path.getsize(QUANTIZED_GGUF) / (1024 * 1024)
        print(f"\n✅ Quantized GGUF: {QUANTIZED_GGUF} ({size_mb:.1f} MB)")
    else:
        print("❌ Quantization failed.")
else:
    print("⚠️ Could not find or build llama-quantize.")
    print("You can quantize manually on the Raspberry Pi:")
    print(f"  llama-quantize {GGUF_OUTPUT} {QUANTIZED_GGUF} {QuantatizeMethod}")

## 10. Create Ollama Modelfile

Generate a `Modelfile` for deploying on Raspberry Pi 5 with Ollama.

In [ ]:
# Determine which GGUF file to use
gguf_for_ollama = QUANTIZED_GGUF if os.path.exists(QUANTIZED_GGUF) else GGUF_OUTPUT
gguf_filename = os.path.basename(gguf_for_ollama)

modelfile_content = f"""# Ollama Modelfile for FunctionGemma Thai Tool Calling (v8)
FROM ./{gguf_filename}

# Gemma 3 chat template
TEMPLATE """{{{{- if .System }}}}<start_of_turn>developer
{{{{ .System }}}}<end_of_turn>
{{{{ end -}}}}{{{{ if .Prompt }}}}<start_of_turn>user
{{{{ .Prompt }}}}<end_of_turn>
{{{{ end -}}}}<start_of_turn>model
{{{{ .Response }}}}<end_of_turn>"""

# Parameters for Raspberry Pi 5
PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER top_k 40
PARAMETER num_predict 256
PARAMETER stop "<end_of_turn>"
PARAMETER stop "<eos>"

# NOTE: No SYSTEM prompt here — tool definitions should be passed
# via the API tools parameter, not hardcoded in the Modelfile.
"""

modelfile_path = "./Modelfile"
with open(modelfile_path, "w") as f:
    f.write(modelfile_content)

print(f"✅ Ollama Modelfile saved to {modelfile_path}")
print()
print("📋 Deployment instructions for Raspberry Pi 5 / LM Studio:")
print("─" * 50)
print(f"1. Copy {gguf_filename} to your device")
print(f"2. In LM Studio: load the GGUF and use the chat API with tools")
print(f'3. Test: POST /v1/chat/completions with tools=[...] and messages=[...]')
print()
print("Generated Modelfile:")
print("─" * 50)
print(modelfile_content)


## 11. Inference Test

Test the fine-tuned model with Thai prompts to verify tool calling capabilities.

In [ ]:
# Load the merged model for inference
if DEVICE == 'cuda':
    inference_model = AutoModelForCausalLM.from_pretrained(
        MERGED_DIR,
        device_map="auto",
        torch_dtype=torch.float16,
    )
else:
    inference_model = AutoModelForCausalLM.from_pretrained(
        MERGED_DIR,
        torch_dtype=torch.float32 if DEVICE == 'mps' else torch.float32,
    )
    inference_model = inference_model.to(DEVICE)

inference_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)

inference_model.eval()
print(f"✅ Inference model loaded on {next(inference_model.parameters()).device}")

In [ ]:
def run_inference(prompt, model, tokenizer, tool_schemas=None, max_new_tokens=256):
    """
    Run inference using the same format as the server.
    No manual developer message — template adds it.
    """
    messages = [{"role": "user", "content": prompt}]

    if tool_schemas:
        input_text = processor.apply_chat_template(
            messages, tools=tool_schemas,
            tokenize=False, add_generation_prompt=True,
        )
    else:
        input_text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )

    device = next(model.parameters()).device
    inputs = tokenizer(input_text, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, temperature=0.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][len(inputs["input_ids"][0]):]
    result = tokenizer.decode(generated, skip_special_tokens=False)
    return result


# ═══════════════════════════════════════════════════════════════
# Test: results should be JSON!
# Tool call: {"tool_call": {"name": ..., "arguments": ...}}
# Response:  {"response": "Thai text"}
# ═══════════════════════════════════════════════════════════════
HA_TOOL_NAMES = list(ha_tool_descriptions.keys())
HA_SCHEMAS = names_to_schemas(HA_TOOL_NAMES)

test_cases = [
    # --- Should output {"tool_call": ...} ---
    {"prompt": "วันนี้อากาศเป็นยังไงที่กรุงเทพ", "schemas": CUSTOM_TOOL_SCHEMAS, "source": "🔧 weather"},
    {"prompt": "หาร้านกาแฟใกล้ๆ", "schemas": CUSTOM_TOOL_SCHEMAS, "source": "🔧 location"},
    {"prompt": "เปิดไฟห้องนอน", "schemas": HA_SCHEMAS, "source": "🏠 HA"},
    # --- Should output {"response": ...} ---
    {"prompt": "สวัสดีครับ เป็นอย่างไรบ้าง", "schemas": CUSTOM_TOOL_SCHEMAS, "source": "💬 chat+tools"},
    {"prompt": "15 คูณ 23 เท่ากับเท่าไหร่", "schemas": CUSTOM_TOOL_SCHEMAS, "source": "💬 chat+tools"},
    {"prompt": "สวัสดีคร๊าบ", "schemas": None, "source": "💬 no tools"},
]

print("🇹🇭 Thai Inference Test (v9 — JSON Output Format)")
print("=" * 60)
for tc in test_cases:
    print(f"\n[{tc['source']}] 📝 {tc['prompt']}")
    response = run_inference(tc["prompt"], inference_model, inference_tokenizer, tool_schemas=tc["schemas"])
    print(f"  🤖 Output: {response}")
    # Check output format
    try:
        parsed = json.loads(response.replace('<end_of_turn>', '').strip())
        if 'tool_call' in parsed:
            print(f"  ✅ Tool call: {parsed['tool_call']['name']}")
        elif 'response' in parsed:
            print(f"  ✅ Response: {parsed['response'][:80]}")
    except:
        print(f"  ⚠️ Not valid JSON")
    print("─" * 60)


## Summary

### v9 — JSON output format (root cause fix)

The model's built-in Jinja template expects JSON:
- Tool call: `{"tool_call": {"name": "func", "arguments": {...}}}`
- Response: `{"response": "Thai text"}`

v9 trains with this exact format — works with LM Studio, llama-server, Ollama.

### Test with Postman:
```json
POST /v1/chat/completions
{
    "model": "functiongemma-thai",
    "tools": [{"type": "function", "function": {"name": "get_current_weather", ...}}],
    "messages": [{"role": "user", "content": "วันนี้อากาศเป็นยังไง"}]
}
```


## Upload Model to HuggingFaceHub

In [ ]:
from huggingface_hub import HfApi, login

# Step 1: Login to Hugging Face
login(token=HF_TOKEN)

# Step 2: Create repo and upload
api = HfApi()

REPO_ID = "aonaon/functiongemma-homeassistant-thai-gguf"  # ← change this

# Create repo if not exists
api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    exist_ok=True,  # won't fail if already exists
    private=False,  # set True if you want private
)
print(f"✅ Repo ready: https://huggingface.co/{REPO_ID}")

# Step 3: Upload GGUF files
gguf_files = [
    "./functiongemma-thai-toolcall-fp16.gguf",
    "./functiongemma-thai-toolcall-Q4_K_M.gguf",
    "./functiongemma-thai-toolcall-Q8_0.gguf",
]

for gguf_file in gguf_files:
    filename = gguf_file.split("/")[-1]
    print(f"📤 Uploading {filename}...")
    api.upload_file(
        path_or_fileobj=gguf_file,
        path_in_repo=filename,
        repo_id=REPO_ID,
        repo_type="model",
    )
    print(f"✅ Uploaded {filename}")